<a href="https://colab.research.google.com/github/cgm2179/indoor-walk-test/blob/main/Physics%20Engine/3D%20Map%20Physics/SIM%20V1%203D/phase_b3_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIM V1 3D — Phase B: generate the surrogate dataset

**Runtime → CPU** (SceneV3 is NumPy / scikit-fmm; a GPU runtime wastes quota).

**Run all checklist**
1. Confirm Drive has this repo under `MyDrive/indoor-walk-test-main/` (including `Physics Engine/2D/SIM/physics_v2.py`).
2. Leave **`RUN_MODE = "full"`** in the Dataset config cell (default). Use `"smoke"` only to verify wiring.
3. **Runtime → Run all**. Full mode is ~1000 Tx × ~9 s ≈ **2–3 hours**.
4. When it finishes, run **Phase C** against the same `surrogate_ds/` folder.

Do not train Phase C on a smoke dataset and expect ≤5 dB — that path is what produced ~15 dB RMSE.


In [ ]:
#@title Mount Drive, install deps, load the scene
import os, sys, json, time, glob
from pathlib import Path
import numpy as np

# GitHub zip on Drive → …/indoor-walk-test-main/…  (edit if your folder name differs)
ROOT = "/content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D"  #@param {type:"string"}

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

ROOT = str(Path(ROOT).expanduser().resolve())
if not Path(ROOT, "engine_3d.py").is_file():
    raise FileNotFoundError(f"ROOT is not SIM V1 3D/: {ROOT}")

# scikit-fmm (eikonal) — Colab does not ship it
try:
    import skfmm  # noqa: F401
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-fmm"], check=True)

# physics_v2 must be importable before engine_3d → physics_3d
_phys2 = Path(ROOT).parent.parent / "2D" / "SIM"
_pv2 = _phys2 / "physics_v2.py"
if not _pv2.is_file():
    _pv2 = Path(ROOT) / "physics_v2.py"
    _phys2 = Path(ROOT)
if not _pv2.is_file():
    raise FileNotFoundError(
        "Missing physics_v2.py.\n"
        f"Expected {_phys2.parent.parent / '2D' / 'SIM' / 'physics_v2.py'} "
        f"or {Path(ROOT) / 'physics_v2.py'}")
sys.path.insert(0, str(_phys2))
sys.path.insert(0, ROOT)
print("physics_v2 →", _pv2)

import dataset_3d as D
from engine_3d import load_scene

scene, man = load_scene(ROOT)
M       = np.load(f"{ROOT}/material_grid.npy")
inside  = np.load(f"{ROOT}/inside_mask.npy")
norm    = D.load_norm(man)
OUT     = f"{ROOT}/dataset"; os.makedirs(OUT, exist_ok=True)

print("grid          ", M.shape, f"({int(np.prod(M.shape)):,} voxels)")
print("interior      ", f"{int(inside.sum()):,} voxels)")
print("scene_sha     ", D.scene_sha(M))
print("bands (MHz)   ", man["freqs_mhz"])
print("norm window   ", f"[{norm.pl_min_db:.0f}, {norm.pl_max_db:.0f}] dB",
      f"| tau ceiling {norm.tau_max_ns:.0f} ns")


## Preflight gate — do not skip this

The target is `clip(PL, 40, 170) dB`. Before M2, direct path loss ran to **1,700+ dB**
because `SceneV3` had no saturating obstruction model. That is fixed: `pathloss_maps`
applies `sat_obs` to the direct crossing loss (`physics.use_satobs`, default on), so
interior clip fractions sit well under the 35 % gate.

**Keep running this cell.** It is the regression tripwire — if someone turns `use_satobs`
off or the cap regresses, generation raises above 35 % instead of baking a flat-plateau
dataset. Raising `CLIP_LIMIT` to get past a real failure is still the one thing that
guarantees a worthless model.


In [ ]:
#@title Preflight: how much of the target saturates the clip ceiling?
CLIP_LIMIT = 0.35      #@param {type:"number"}  max tolerable clipped fraction
N_PROBE    = 4         #@param {type:"integer"}

rep = D.clip_report(scene, man, inside, norm, n_probe=N_PROBE, seed=0)
print(f"clip ceiling {rep['pl_max_db']:.0f} dB, averaged over {rep['n_probe']} random Tx\n")
print(f"{'band (MHz)':>12} {'clipped':>9} {'median PL':>11}")
for f in sorted(rep["clipped_fraction"]):
    print(f"{f:12.0f} {rep['clipped_fraction'][f]*100:8.1f}% {rep['median_pl_db'][f]:10.1f} dB")

worst = rep["worst_clipped_fraction"]
print(f"\nworst clipped fraction: {worst*100:.1f}%  (limit {CLIP_LIMIT*100:.0f}%)")
if worst > CLIP_LIMIT:
    raise SystemExit(
        f"PREFLIGHT FAILED — {worst*100:.1f}% of interior voxels saturate the {rep['pl_max_db']:.0f} dB "
        "ceiling.\nThe dataset would be mostly a constant. Check SceneV3.use_satobs / sat_obs "
        "(M2 saturating-obstruction) — see PLAN_3D_SIM.md.")
print("preflight OK")


## Config

`RUN_MODE = "smoke"` solves a handful of Tx positions end-to-end in about a minute — use it once to verify Drive paths and the solver.

`RUN_MODE = "full"` (the default) is the real dataset Phase C needs. It is resumable: re-running skips shards that already have a `_meta.json`.


In [ ]:
#@title Dataset config
# RUN_MODE="full" → ~1000 Tx × ~9 s ≈ 2–3 h on Colab CPU (what you need for ≤5 dB).
# RUN_MODE="smoke" → 16 Tx, ~2 min (plumbing check only).
RUN_MODE      = "full"   #@param ["full", "smoke"]
N_POSITIONS   = 1000     #@param {type:"integer"}
SHARD_POS     = 25       #@param {type:"integer"}
TRAIN_BANDS   = [619.0, 1935.0, 2442.0, 3500.0, 5500.0, 6125.0]
TAU_BAND_MHZ  = 3500.0

SMOKE = (RUN_MODE == "smoke")
if SMOKE:
    N_POSITIONS, SHARD_POS = 16, 8

splits_file = f"{OUT}/splits.json"
sp = json.load(open(splits_file))
if sp.get("scene_sha") != D.scene_sha(M):
    raise SystemExit(
        f"splits.json scene_sha {sp.get('scene_sha')} != live {D.scene_sha(M)} — "
        "regenerate splits or restore the matching grid.")

positions = np.asarray(sp["positions"], float)
assert len(positions) >= 1
N_POSITIONS = min(N_POSITIONS, len(positions))
if N_POSITIONS < len(positions) and not SMOKE:
    print(f"{len(positions)} positions available; using first {N_POSITIONS}")

# Persist the run budget into splits so Phase C can refuse a smoke-sized train.
sp_out = dict(sp)
sp_out["train_bands_mhz"] = TRAIN_BANDS
sp_out["n_positions_requested"] = int(N_POSITIONS)
sp_out["run_mode"] = RUN_MODE
json.dump(sp_out, open(splits_file, "w"), indent=1)

mb = int(np.prod(M.shape)) * 2 / 1e6   # fp16 MB per volume
print(f"{N_POSITIONS} positions × {len(TRAIN_BANDS)} bands = {N_POSITIONS*len(TRAIN_BANDS)} samples  [{RUN_MODE}]")
print(f"projected size: PL {N_POSITIONS*len(TRAIN_BANDS)*mb/1000:.1f} GB "
      f"+ tau {N_POSITIONS*mb/1000:.1f} GB")
if SMOKE:
    print("NOTE: smoke dataset cannot hit the ≤5 dB target — re-run with RUN_MODE='full'.")


## Generate shards

Resumable: a shard is written to `.tmp` names and renamed only once all three files are
complete, so a disconnected runtime never leaves a half shard that a later run would trust.
Re-run this cell to pick up where it stopped.


In [ ]:
#@title Solve and write shards (re-runnable)
n_shards = int(np.ceil(N_POSITIONS / SHARD_POS))
band_idx = [scene.band_index(f) for f in TRAIN_BANDS]
t_start, done = time.time(), 0

for s in range(n_shards):
    if D.shard_complete(OUT, s):
        print(f"shard {s:03d}: complete, skipping"); continue
    p0, p1 = s * SHARD_POS, min((s + 1) * SHARD_POS, N_POSITIONS)
    pl_rows, tau_rows = [], []
    m_tx, m_f, m_ff, m_pos, m_taurow = [], [], [], [], []

    for pi in range(p0, p1):
        tx = tuple(float(v) for v in positions[pi])
        PL = scene.pathloss_maps(tx)                     # one geometric pass, all bands
        T  = scene.arrival_time(tx, TAU_BAND_MHZ)
        tau_rows.append(norm.tau_to_norm(T))
        for f, bi in zip(TRAIN_BANDS, band_idx):
            pl_rows.append(norm.pl_to_norm(PL[bi]))
            m_tx.append(positions[pi]); m_f.append(f)
            m_ff.append(norm.freq_feature(f)); m_pos.append(pi)
            m_taurow.append(len(tau_rows) - 1)
        done += 1

    D.write_shard(OUT, s, np.stack(pl_rows), np.stack(tau_rows), dict(
        tx=np.array(m_tx, np.int16), freq_mhz=np.array(m_f, np.float32),
        freq_feat=np.array(m_ff, np.float32), pos_id=np.array(m_pos, np.int32),
        tau_row=np.array(m_taurow, np.int32), scene_sha=D.scene_sha(M),
        bands_mhz=np.array(TRAIN_BANDS, np.float32)))

    rate = (time.time() - t_start) / max(done, 1)
    left = (N_POSITIONS - p1) * rate
    print(f"shard {s+1:3d}/{n_shards}  positions {p0}-{p1-1}  "
          f"{rate:.1f} s/pos  ETA {left/60:.0f} min")

print(f"\ndone: {len(D.list_shards(OUT))} shards in {(time.time()-t_start)/60:.1f} min")


## Sanity check

Reads a shard back **through the memmap path Phase C will use**, denormalizes, and plots a
horizontal slice at the Tx height. Loss should be low at the star and rise with distance and
through walls. The printed round-trip error is the fp16 quantization floor, so it should sit
near 0.01 dB, not near 1 dB.


In [ ]:
#@title Read one sample back and plot it
import matplotlib.pyplot as plt

pl, tau, meta = D.open_shard(OUT, 0)
i  = 0
tx = meta["tx"][i]; f = float(meta["freq_mhz"][i])
vol = norm.norm_to_pl(np.asarray(pl[i], np.float32))
tv  = norm.norm_to_tau_ns(np.asarray(tau[int(meta["tau_row"][i])], np.float32))
iy  = int(tx[1])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, img, title, cb in (
        (ax[0], np.where(inside, vol, np.nan)[:, iy, :].T, f"PL @ {f:.0f} MHz, iy={iy}", "dB"),
        (ax[1], np.where(inside, tv, np.nan)[:, iy, :].T, "eikonal arrival", "ns")):
    im = a.imshow(img, origin="lower", cmap="viridis")
    a.plot(tx[0], tx[2], "r*", ms=14, mec="white"); a.set_title(title)
    fig.colorbar(im, ax=a, label=cb)
plt.tight_layout(); plt.show()

ref = scene.pathloss_maps(tuple(float(v) for v in tx))[scene.band_index(f)]
err = np.abs(np.clip(ref, norm.pl_min_db, norm.pl_max_db) - vol)[inside].max()
print(f"Tx {tx.tolist()}  interior PL {vol[inside].min():.1f} - {vol[inside].max():.1f} dB")
print(f"tau max {tv[inside].max():.0f} ns")
print(f"fp16 round-trip error vs a fresh solve: {err:.4f} dB")
assert err < 0.2, "shard does not reproduce the engine - check norm/band indexing"
print("\nDataset ready. Next: phase_c3_train_colab.ipynb")
